In [1]:
%cd /drive2/ryusejong/LFF
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
import json 
import time 
import re
import random
import types
import math
import numpy as np 
from tqdm.auto import tqdm
from util.utils import set_seed, read_data, save_result, get_answer_from_text, chat_huggingface, chat_huggingface_with_hidden_states, construct_conversation
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel


seed = 42
set_seed(seed)

/drive2/ryusejong/LFF


/drive2/ryusejong/miniconda3/envs/llm1/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
### laod prm model
prm_path = "UW-Madison-Lee-Lab/Llama-PRM800K"
device = "cuda:0" if torch.cuda.is_available() else "cpu"

candidate_tokens = [12, 10]
prm_tokenizer = AutoTokenizer.from_pretrained(prm_path)
prm_tokenizer.pad_token = prm_tokenizer.eos_token
prm_tokenizer.padding_side = 'left' 
prm_tokenizer.truncation_side = 'left'
    
prm = AutoModelForCausalLM.from_pretrained(
    prm_path,
    torch_dtype=torch.bfloat16,
    device_map=device
)
prm.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:06<00:00,  1.53s/it]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
    (rotary_

In [4]:
### original score
with torch.no_grad():
    prm_input = torch.tensor([prm_tokenizer.encode(prm_input_text)]).to(prm.device)
    prm_logits = prm(prm_input).logits[:,:,candidate_tokens]
    #print(logits.shape)
    prm_scores = prm_logits.softmax(dim=-1)[:,:,1]
    #print(scores.shape)
    step_scores = prm_scores[prm_input == 23535]
    step_probs  = step_scores.tolist()
    
print(f"Step scores: {step_scores}")

Step scores: tensor([0.9492, 0.6094, 0.9648, 0.9727, 0.9414, 0.9414, 0.9688],
       device='cuda:0', dtype=torch.bfloat16)


In [13]:
### incorrect case
question = "Bryan starts exercising at home during quarantine. To start, he decides to do 3 sets of 15 push-ups each. Near the end of the third set, he gets tired and does 5 fewer push-ups. How many push-ups did he do in total? Explain your reasoning step-by-step. Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response."

reasoning = "Let's break it down step-by-step!\n\n1. Bryan starts with 3 sets of 15 push-ups each. So, he does 3 x 15 = 45 push-ups in the first two sets.\n2. In the third set, he does 5 fewer push-ups than usual. So, he does 15 - 5 = 10 push-ups in the third set.\n3. To find the total number of push-ups, we add the number of push-ups in the first two sets (45) to the number of push-ups in the third set (10).\n\n45 + 10 = 55\n\n## 55 ##\n\nSo, Bryan did a total of 55 push-ups."

### incorrect (miss condition) step index : 1
reasoning_steps = [l.strip() for l in reasoning.split("\n") if l.strip()]
print(f"# steps: {len(reasoning_steps)}")
print("-"*50)
prm_input_text = question + ' \n\n' + ' \n\n\n\n'.join(reasoning_steps) + ' \n\n\n\n'

### original score
with torch.no_grad():
    prm_input = torch.tensor([prm_tokenizer.encode(prm_input_text)]).to(prm.device)
    prm_logits = prm(prm_input).logits[:,:,candidate_tokens]
    #print(logits.shape)
    prm_scores = prm_logits.softmax(dim=-1)[:,:,1]
    #print(scores.shape)
    step_scores = prm_scores[prm_input == 23535]
    step_probs  = step_scores.tolist()
    
print(f"Step scores: {step_scores}")
print("-"*50)
### score of question + each step
### train 29
for i, step in enumerate(reasoning_steps):
    if i == 0:
        prm_input_text = question + ' \n\n' + step + ' \n\n\n\n'
    else:
        prm_input_text = question + ' \n\n' + reasoning_steps[i-1] + ' \n\n\n\n' + step + ' \n\n\n\n'
    with torch.no_grad():
        prm_input = torch.tensor([prm_tokenizer.encode(prm_input_text)]).to(prm.device)
        prm_logits = prm(prm_input).logits[:,:,candidate_tokens]
        #print(logits.shape)
        prm_scores = prm_logits.softmax(dim=-1)[:,:,1]
        #print(scores.shape)
        step_scores = prm_scores[prm_input == 23535]
        step_probs  = step_scores.tolist()
    #print(question)
    print(f"Step: {step} \nScore: {step_probs}")

# steps: 7
--------------------------------------------------
Step scores: tensor([0.9492, 0.6094, 0.9648, 0.9727, 0.9414, 0.9414, 0.9688],
       device='cuda:0', dtype=torch.bfloat16)
--------------------------------------------------
Step: Let's break it down step-by-step! 
Score: [0.94921875]
Step: 1. Bryan starts with 3 sets of 15 push-ups each. So, he does 3 x 15 = 45 push-ups in the first two sets. 
Score: [0.94921875, 0.609375]
Step: 2. In the third set, he does 5 fewer push-ups than usual. So, he does 15 - 5 = 10 push-ups in the third set. 
Score: [0.546875, 0.96484375]
Step: 3. To find the total number of push-ups, we add the number of push-ups in the first two sets (45) to the number of push-ups in the third set (10). 
Score: [0.7734375, 0.6484375]
Step: 45 + 10 = 55 
Score: [0.20703125, 0.5625]
Step: ## 55 ## 
Score: [0.11865234375, 0.4296875]
Step: So, Bryan did a total of 55 push-ups. 
Score: [0.2197265625, 0.62890625]


In [14]:
### incorrect case
question = "Reina has three times the number of counters and four times the number of marbles as Kevin. If Kevin has 40 counters and 50 marbles, calculate the number of counters and marbles that Reina has? Explain your reasoning step-by-step. Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response."

reasoning = "Let's break it down step by step!\n\n1. Reina has three times the number of counters as Kevin.\nKevin has 40 counters, so Reina has 3 x 40 = 120 counters.\n\n2. Reina has four times the number of marbles as Kevin.\nKevin has 50 marbles, so Reina has 4 x 50 = 200 marbles.\n\n## 120 counters and 200 marbles ##\n\nSo, Reina has 120 counters and 200 marbles."

### incorrect (miss condition) step index : 1
reasoning_steps = [l.strip() for l in reasoning.split("\n") if l.strip()]
print(f"# steps: {len(reasoning_steps)}")
print("-"*50)
prm_input_text = question + ' \n\n' + ' \n\n\n\n'.join(reasoning_steps) + ' \n\n\n\n'

### original score
with torch.no_grad():
    prm_input = torch.tensor([prm_tokenizer.encode(prm_input_text)]).to(prm.device)
    prm_logits = prm(prm_input).logits[:,:,candidate_tokens]
    #print(logits.shape)
    prm_scores = prm_logits.softmax(dim=-1)[:,:,1]
    #print(scores.shape)
    step_scores = prm_scores[prm_input == 23535]
    step_probs  = step_scores.tolist()
    
print(f"Step scores: {step_scores}")
print("-"*50)
### score of question + each step
### train 29
for i, step in enumerate(reasoning_steps):
    if i == 0:
        prm_input_text = question + ' \n\n' + step + ' \n\n\n\n'
    else:
        prm_input_text = question + ' \n\n' + reasoning_steps[i-1] + ' \n\n\n\n' + step + ' \n\n\n\n'
    with torch.no_grad():
        prm_input = torch.tensor([prm_tokenizer.encode(prm_input_text)]).to(prm.device)
        prm_logits = prm(prm_input).logits[:,:,candidate_tokens]
        #print(logits.shape)
        prm_scores = prm_logits.softmax(dim=-1)[:,:,1]
        #print(scores.shape)
        step_scores = prm_scores[prm_input == 23535]
        step_probs  = step_scores.tolist()
    #print(question)
    print(f"Step: {step} \nScore: {step_probs}")

# steps: 7
--------------------------------------------------
Step scores: tensor([0.9453, 0.9844, 0.9922, 0.9961, 0.9961, 0.7734, 0.9336],
       device='cuda:0', dtype=torch.bfloat16)
--------------------------------------------------
Step: Let's break it down step by step! 
Score: [0.94140625]
Step: 1. Reina has three times the number of counters as Kevin. 
Score: [0.9453125, 0.984375]
Step: Kevin has 40 counters, so Reina has 3 x 40 = 120 counters. 
Score: [0.94921875, 0.9921875]
Step: 2. Reina has four times the number of marbles as Kevin. 
Score: [0.9765625, 0.96875]
Step: Kevin has 50 marbles, so Reina has 4 x 50 = 200 marbles. 
Score: [0.53515625, 0.94140625]
Step: ## 120 counters and 200 marbles ## 
Score: [0.78125, 0.341796875]
Step: So, Reina has 120 counters and 200 marbles. 
Score: [0.4453125, 0.796875]


In [12]:
### incorrect case
question = "Jane picked 64 apples at the orchard. She sorted them into 4 different baskets to send to her friends. When Jane wasn't looking her sister took 3 apples from each of the baskets. How many apples are in each basket now? Explain your reasoning step-by-step. Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response."

reasoning = "Let's break this problem down step by step!\n\n1. Jane picked 64 apples and sorted them into 4 baskets. To find out how many apples are in each basket initially, we can divide the total number of apples (64) by the number of baskets (4):\n\n64 \u00f7 4 = 16\n\nSo, each basket initially had 16 apples.\n\n2. Jane's sister took 3 apples from each of the baskets. To find out how many apples are left in each basket, we need to subtract 3 from the initial number of apples in each basket (16):\n\n16 - 3 = 13\n\nSo, each basket now has 13 apples.\n\n## 13 ##\n\nTherefore, there are 13 apples in each basket now. Jane's sister took 3 apples from each basket, leaving 13 apples in each basket."

### incorrect (miss condition) step index : 1
reasoning_steps = [l.strip() for l in reasoning.split("\n") if l.strip()]
print(f"# steps: {len(reasoning_steps)}")
print("-"*50)
prm_input_text = question + ' \n\n' + ' \n\n\n\n'.join(reasoning_steps) + ' \n\n\n\n'

### original score
with torch.no_grad():
    prm_input = torch.tensor([prm_tokenizer.encode(prm_input_text)]).to(prm.device)
    prm_logits = prm(prm_input).logits[:,:,candidate_tokens]
    #print(logits.shape)
    prm_scores = prm_logits.softmax(dim=-1)[:,:,1]
    #print(scores.shape)
    step_scores = prm_scores[prm_input == 23535]
    step_probs  = step_scores.tolist()
    
print(f"Step scores: {step_scores}")
print("-"*50)
### score of question + each step
### train 29
for i, step in enumerate(reasoning_steps):
    if i == 0:
        prm_input_text = question + ' \n\n' + step + ' \n\n\n\n'
    else:
        prm_input_text = question + ' \n\n' + reasoning_steps[i-1] + ' \n\n\n\n' + step + ' \n\n\n\n'
    with torch.no_grad():
        prm_input = torch.tensor([prm_tokenizer.encode(prm_input_text)]).to(prm.device)
        prm_logits = prm(prm_input).logits[:,:,candidate_tokens]
        #print(logits.shape)
        prm_scores = prm_logits.softmax(dim=-1)[:,:,1]
        #print(scores.shape)
        step_scores = prm_scores[prm_input == 23535]
        step_probs  = step_scores.tolist()
    #print(question)
    print(f"Step: {step} \nScore: {step_probs}")

# steps: 9
--------------------------------------------------
Step scores: tensor([0.9570, 0.9258, 0.9922, 0.9961, 0.9883, 0.9922, 0.9922, 0.9609, 0.8750],
       device='cuda:0', dtype=torch.bfloat16)
--------------------------------------------------
Step: Let's break this problem down step by step! 
Score: [0.95703125]
Step: 1. Jane picked 64 apples and sorted them into 4 baskets. To find out how many apples are in each basket initially, we can divide the total number of apples (64) by the number of baskets (4): 
Score: [0.95703125, 0.92578125]
Step: 64 ÷ 4 = 16 
Score: [0.8359375, 0.984375]
Step: So, each basket initially had 16 apples. 
Score: [0.89453125, 0.9921875]
Step: 2. Jane's sister took 3 apples from each of the baskets. To find out how many apples are left in each basket, we need to subtract 3 from the initial number of apples in each basket (16): 
Score: [0.94140625, 0.86328125]
Step: 16 - 3 = 13 
Score: [0.396484375, 0.67578125]
Step: So, each basket now has 13 apples. 

In [19]:
### incorrect case
question = "Bryan starts exercising at home during quarantine. To start, he decides to do 3 sets of 15 push-ups each. Near the end of the third set, he gets tired and does 5 fewer push-ups. How many push-ups did he do in total? Explain your reasoning step-by-step. Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response."

reasoning = "Let's break it down step-by-step!\n\n1. Bryan starts with 3 sets of 15 push-ups each. So, he does 3 x 15 = 45 push-ups in the first two sets.\n2. In the third set, he does 5 fewer push-ups than usual. So, he does 15 - 5 = 10 push-ups in the third set.\n3. To find the total number of push-ups, we add the number of push-ups in the first two sets (45) to the number of push-ups in the third set (10).\n\n45 + 10 = 55\n\n## 55 ##\n\nSo, Bryan did a total of 55 push-ups."

### incorrect (miss condition) step index : 1
reasoning_steps = [l.strip() for l in reasoning.split("\n") if l.strip()]
print(f"# steps: {len(reasoning_steps)}")
print("-"*50)
prm_input_text = question + ' \n\n' + ' \n\n\n\n'.join(reasoning_steps) + ' \n\n\n\n'

### original score
with torch.no_grad():
    prm_input = torch.tensor([prm_tokenizer.encode(prm_input_text)]).to(prm.device)
    prm_logits = prm(prm_input).logits[:,:,candidate_tokens]
    #print(logits.shape)
    prm_scores = prm_logits.softmax(dim=-1)[:,:,1]
    #print(scores.shape)
    step_scores = prm_scores[prm_input == 23535]
    step_probs  = step_scores.tolist()
    
print(f"Step scores: {step_scores}")
print("-"*50)
### score of question + each step
### train 29
for i, step in enumerate(reasoning_steps):
    prm_input_text = question + ' \n\n' + ' \n\n\n\n'.join(reasoning_steps[:i+1]) + ' \n\n\n\n' + 'This steps follow the conditions of the problem.' + ' \n\n\n\n'
    with torch.no_grad():
        prm_input = torch.tensor([prm_tokenizer.encode(prm_input_text)]).to(prm.device)
        prm_logits = prm(prm_input).logits[:,:,candidate_tokens]
        #print(logits.shape)
        prm_scores = prm_logits.softmax(dim=-1)[:,:,1]
        #print(scores.shape)
        step_scores = prm_scores[prm_input == 23535]
        step_probs  = step_scores.tolist()
    #print(question)
    print(f"Step: {step} \nScore: {step_probs}")

# steps: 7
--------------------------------------------------
Step scores: tensor([0.9492, 0.6094, 0.9648, 0.9727, 0.9414, 0.9414, 0.9688],
       device='cuda:0', dtype=torch.bfloat16)
--------------------------------------------------
Step: Let's break it down step-by-step! 
Score: [0.94921875, 0.84375]
Step: 1. Bryan starts with 3 sets of 15 push-ups each. So, he does 3 x 15 = 45 push-ups in the first two sets. 
Score: [0.94921875, 0.609375, 0.71875]
Step: 2. In the third set, he does 5 fewer push-ups than usual. So, he does 15 - 5 = 10 push-ups in the third set. 
Score: [0.94921875, 0.60546875, 0.9609375, 0.74609375]
Step: 3. To find the total number of push-ups, we add the number of push-ups in the first two sets (45) to the number of push-ups in the third set (10). 
Score: [0.94921875, 0.609375, 0.96484375, 0.97265625, 0.8203125]
Step: 45 + 10 = 55 
Score: [0.94921875, 0.609375, 0.96484375, 0.97265625, 0.94140625, 0.85546875]
Step: ## 55 ## 
Score: [0.94921875, 0.609375, 0.964843